In [1]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/022026/Data/MEDS_MDS/data/tuning/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.1) and mlflow-skinny (2.22.1) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,30,1990-04-12 00:00:00,DOB,NaN
1,30,2016-12-30 00:00:00,D/DS934,NaN
2,30,2016-12-30 20:51:00,ADMISSION_ADT,NaN
3,30,2016-12-30 20:51:00,MOVE_ADT,NaN
4,30,2016-12-30 20:51:00,^AFSNIT_ADT/,NaN
5,30,NaT,GENDER//Mand,NaN
6,195,2013-05-25 00:00:00,DOB,NaN
7,195,2018-09-20 00:00:00,D/DS014A,NaN
8,195,2018-09-20 17:15:00,ADMISSION_ADT,NaN
9,195,2018-09-20 17:15:00,MOVE_ADT,NaN


In [2]:
len(df)

45781299

In [3]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 221803
The patients has M-medication Codes: 150771
The patients has D-diagnosis Codes: 221755
The patients has P-Procedure Codes: 0
The patients has S-SKS Codes: 52216


In [4]:
subject_counts = df['subject_id'].value_counts()


In [5]:
subject_counts

684625     53884
132804     49438
1968194    47853
856501     47594
1394370    45693
           ...  
242591         2
1436020        2
1085023        2
1831164        2
2138700        2
Name: subject_id, Length: 221803, dtype: int64

In [6]:
p_Num = df[df['code'].str.startswith('S/', na=False)]

In [7]:
p_Num

,subject_id,time,code,numeric_value
175,575,2021-10-22 23:59:00,S/KQBA10B,NaN
940,775,2018-05-31 23:59:00,S/KCJE20,NaN
977,775,2018-08-27 23:59:00,S/KJFB21,NaN
2882,1225,2017-11-29 23:59:00,S/KMCA10,NaN
4426,1405,2020-07-20 23:59:00,S/KKFC00,NaN
...,...,...,...,...
45764340,2216909,2021-04-02 23:59:00,S/KJEA01,NaN
45764621,2217234,2019-04-02 23:59:00,S/KNHJ82,NaN
45764819,2217319,2020-01-27 23:59:00,S/KEMB15 KEMB30,NaN
45766732,2217479,2016-07-19 23:59:00,S/KKBE12,NaN


In [8]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('S/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('S/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only porcedure code: ", only_p_ids_to_exclude)


Number of patients with only porcedure code:  []


In [9]:
df_filtered = df[~df['code'].str.startswith('S/', na=False)]

In [10]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [11]:
subject_counts_MDS

684625     53884
132804     49438
1968194    47853
856501     47594
1394370    45693
           ...  
219083         2
907736         2
2138700        2
964425         2
1869208        2
Name: subject_id, Length: 221803, dtype: int64

In [14]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [15]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [16]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [17]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [18]:
comparison_df

,subject_id,original_count,new_count,difference
1651,1719866,3101,3072,29
185,1966150,8449,8424,25
10171,712426,992,970,22
2562,992199,2469,2449,20
765,1429160,4473,4455,18
...,...,...,...,...
102195,2199250,52,52,0
102198,239138,52,52,0
102199,159993,52,52,0
102200,1683276,52,52,0


In [19]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 169587


In [20]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [21]:
print(most_changed.head(10))


       subject_id  original_count  new_count  difference  abs_diff
1651      1719866            3101       3072          29        29
185       1966150            8449       8424          25        25
10171      712426             992        970          22        22
2562       992199            2469       2449          20        20
765       1429160            4473       4455          18        18
5310      1520096            1585       1567          18        18
10158     1394776             994        977          17        17
1938       896805            2865       2852          13        13
86         343352           11875      11862          13        13
4825      1166108            1693       1680          13        13


In [22]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [23]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDS codes',
        'new_count': 'MD codes'
    }
)


In [24]:
lowest_new_count_patients

,subject_id,MDS codes,MD codes,difference,abs_diff
212746,1864799,4,3,1,1
213077,609609,4,3,1,1


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [25]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

221803

In [26]:
len(df_filtered)

45705092

In [27]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('S/', na=False)].copy()
print("kept rows MDP:", len(df_filtered), " / total:", len(df))


kept rows MDP: 45705092  / total: 45781299


In [28]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
#N_SHARDS = 45
#df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

#print("rows to write:", len(df_filtered))


In [29]:
import numpy as np
import os

N_SHARDS = 5   #45 for Whole # 36 when we have split
OUT_DIR = "./_tuningMDP_withoutS_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


Done. wrote 5 parquet files into ./_tuningMDP_withoutS_sharded


In [30]:
import pyarrow.parquet as pq

OUT_DIR = "./_tuningMDP_withoutS_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


0.parquet rows: 9108892
1.parquet rows: 9093752
2.parquet rows: 9374059
3.parquet rows: 9150531
4.parquet rows: 8977858
TOTAL rows: 45705092


In [31]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_tuningMDP_withoutS_sharded"
DST_PREFIX = "Zahra/022026/Data/MEDS_MD/data/tuning"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


Local files to upload: 5
Validating arguments.
Arguments validated.
'overwrite' is set to True. Any file already present in the target will be overwritten.
Uploading files from '/mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_tuningMDP_withoutS_sharded' to 'Zahra/022026/Data/MEDS_MD/data/tuning'
Copying 5 files with concurrency set to 5
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_tuningMDP_withoutS_sharded/1.parquet, file 1 out of 5. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/022026/Data/MEDS_MD/data/tuning/1.parquet
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_tuningMDP_withoutS_sharded/3.parquet, file 2 out of 5. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/022026/Data/MED

In [32]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:20])


{'infer_column_types': 'False', 'activity': 'to_path'}
{'infer_column_types': 'False', 'activity': 'to_path', 'activityApp': 'FileDataset'}
Found in datastore: 5
['/0.parquet', '/1.parquet', '/2.parquet', '/3.parquet', '/4.parquet']
